# SupplyMind AI — Model Comparison & Champion Selection

Compare all candidate models on the validation partition, select the champion
according to the RFC, then evaluate that model once on the untouched latest
test partition.

**Selection order:** F1 → recall → ROC-AUC.

In [1]:
# -------------------
# Imports
# -------------------

import json
from pathlib import Path

import pandas as pd

In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Load candidate metrics
# -------------------

candidate_names = [
    "logistic_regression",
    "random_forest",
    "xgboost",
    "hist_gradient_boosting",
]

rows = []

for name in candidate_names:
    metrics_path = (
        REPORT_ROOT / "models" / name / "validation_metrics.json"
    )

    if not metrics_path.exists():
        print(f"Missing metrics for {name}: run its notebook first.")
        continue

    metrics = json.loads(metrics_path.read_text())
    rows.append({"model_name": name, **metrics})

comparison = pd.DataFrame(rows).sort_values(
    ["f1", "recall", "roc_auc"],
    ascending=[False, False, False],
)

comparison

,model_name,accuracy,precision,recall,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,threshold
1,random_forest,0.582515,0.581378,0.988412,0.732125,0.738214,0.833870,280,9581,156,13306,0.30
2,xgboost,0.578828,0.578464,0.996434,0.731985,0.739064,0.834281,86,9775,48,13414,0.27
3,hist_gradient_boosting,0.577241,0.577223,1.000000,0.731949,0.739468,0.834411,1,9860,0,13462,0.23
0,logistic_regression,0.577413,0.577422,0.998886,0.731810,0.740043,0.834546,20,9841,15,13447,0.22


In [4]:
# -------------------
# Champion candidate
# -------------------

if comparison.empty:
    raise RuntimeError("No model results were found.")

champion_name = comparison.iloc[0]["model_name"]
champion_threshold = comparison.iloc[0]["threshold"]

print("Selected champion:", champion_name)
print("Threshold:", champion_threshold)

Selected champion: random_forest
Threshold: 0.3000000000000001


### Conclusion
All four models are statistically tied on F1 and AUC, and all four are close to a majority-class baseline which my EDA predicted, since the features correlate weakly with delay. I selected Random Forest as champion because it retains the most genuine discrimination, the highest true-negative count and precision, rather than collapsing to always predicting 'delayed' like HistGradientBoosting does.